# NISAR field-mean backscatter — 2026 fields

Spatial mean of HHHH, HVHV and HH−HV over the five 2026 fields for every pass,
a time series, and one map figure per pass (rows = fields).

**Data:** small GeoTIFF clips made once on ORCD by `scripts/clip_fields.py`
(see its header). Copy `data/nisar_field_clips/` here and this runs locally.

**To use:** edit only cell 2, then *Run All*. Outputs go to `outputs/`.
After changing only plot settings, re-run cells 6–7.

## 1. Imports

In [ ]:
import re, warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask
from shapely.geometry import Polygon
from shapely.ops import unary_union

warnings.filterwarnings("ignore")
plt.rcParams.update({                                    # large text on every figure
    'font.size': 14, 'axes.titlesize': 17, 'axes.labelsize': 15,
    'xtick.labelsize': 13, 'ytick.labelsize': 13,
    'legend.fontsize': 13, 'legend.title_fontsize': 13, 'figure.titlesize': 20,
})

# Repo root: works whether Jupyter starts in notebooks/ or in the repo root.
ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "scripts").is_dir())
print("repo root:", ROOT)

## 2. Config — edit here

In [ ]:
# ---- inputs (relative to repo root) -----------------------------------------
CLIP_DIR   = "data/nisar_field_clips"          # output of scripts/clip_fields.py
YEAR       = 2026                              # keep only passes from this year
FIELDS_CSV = "final/Fields_2024 - Sheet1.csv"  # Earth Engine polygons per field
GAUGES_CSV = "scripts/gauges_2026.csv"         # gauges, drawn as markers on maps

# ---- the 2026 fields --------------------------------------------------------
# Field name (as in gauges_2026.csv) -> polygon code(s) in FIELDS_CSV.
# Two codes are merged: Way 3 and Way 4 each have one polygon per gauge, ~99%
# overlapping. Dict order = row order on the maps.
FIELDS_2026 = {
    "Way 3":   ["W3-EC", "W3-CS"],
    "Way 4":   ["W4-EC", "W4-CS"],
    "4 East":  ["R8"],
    "South 0": ["S0"],
    "S16":     [],          # not in any Fields_*.csv (2021-2024) -> see MANUAL_POLYGONS
}

# Polygons missing from FIELDS_CSV: name -> [[lon, lat], ...]. Draw on geojson.io
# (satellite basemap) and paste the vertices. No re-clipping needed: the S16 clip
# already covers 800 m around its gauge.
MANUAL_POLYGONS = {
    # "S16": [[-91.8120, 34.8480], [-91.8080, 34.8480], [-91.8080, 34.8520], [-91.8120, 34.8520]],
}

# ---- spatial mean -----------------------------------------------------------
BUFFER_M = 0           # shrink polygons inward before averaging (20-30 drops levee pixels)

# ---- maps -------------------------------------------------------------------
MAP_LOOKS     = 2      # block-average NxN pixels (linear power) before mapping; 1 = 10 m
MAP_DATES     = [0]    # which passes to SHOW inline (indices into `passes`)
SAVE_ALL_MAPS = True   # write one PNG per pass to MAP_DIR
VLIM = {"HHHH": (-25, 0), "HVHV": (-30, -10), "ratio": (0, 15)}   # fixed colour limits, dB

# ---- outputs ----------------------------------------------------------------
OUT_CSV = ROOT / "outputs" / "nisar_field_means_2026.csv"
TS_PNG  = ROOT / "outputs" / "nisar_field_means_2026_timeseries.png"
MAP_DIR = ROOT / "outputs" / "nisar_field_maps_2026"

## 3. Fields, gauges and the clip index

In [ ]:
def parse_ee_polygon(text):
    """Shapely Polygon from an 'ee.Geometry.Polygon([[[lon, lat], ...]])' string."""
    num = r"-?\d+\.\d+"
    pts = re.findall(rf"\[\s*({num})\s*,\s*({num})\s*\]", str(text))
    return Polygon([(float(lon), float(lat)) for lon, lat in pts])


raw = pd.read_csv(ROOT / FIELDS_CSV)
csv_polys = {c.strip(): parse_ee_polygon(t) for c, t in zip(raw["Field code"], raw["Geometry details"])}

rows = []
for name, codes in FIELDS_2026.items():
    if name in MANUAL_POLYGONS:                          # pasted polygon wins
        poly, source = Polygon(MANUAL_POLYGONS[name]), "manual"
    elif codes and all(c in csv_polys for c in codes):
        poly, source = unary_union([csv_polys[c] for c in codes]), "+".join(codes)
    else:
        poly, source = None, "none"
    rows.append({"field": name, "source": source, "geometry": poly})
fields = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
fields["has_polygon"] = fields.geometry.notna()
fields["area_ha"] = fields.to_crs(32615).area / 1e4

gauges = pd.read_csv(ROOT / GAUGES_CSV)
gauges = gpd.GeoDataFrame(gauges, geometry=gpd.points_from_xy(gauges.lon, gauges.lat), crs="EPSG:4326")

# One row per (pass, field) clip; written by scripts/clip_fields.py.
index = pd.read_csv(ROOT / CLIP_DIR / "index.csv", parse_dates=["acq_utc"])
index = index[(index.acq_utc.dt.year == YEAR) & index.file.notna()]
passes = index.drop_duplicates("granule").sort_values("acq_utc").reset_index(drop=True)

print(fields[["field", "source", "has_polygon", "area_ha"]].round(1).to_string(index=False))
print(f"\n{len(passes)} passes, {len(index)} clips in {CLIP_DIR}")

## 4. Helper functions

Clips hold **linear** power with invalid pixels already NaN. Average in linear,
then convert to dB.

In [ ]:
def to_db(lin):
    return 10 * np.log10(lin)


def clip_path(granule, field):
    """GeoTIFF for one pass x field, or None if that field wasn't clipped."""
    hit = index[(index.granule == granule) & (index.field == field)]
    return ROOT / CLIP_DIR / hit.file.iloc[0] if len(hit) else None


def read_clip(path):
    """HHHH, HVHV (linear, NaN = invalid), affine transform, EPSG code."""
    with rasterio.open(path) as src:
        return src.read(1), src.read(2), src.transform, src.crs.to_epsg()


def field_mean(path, polygon):
    """Spatial mean over `polygon` (lon/lat) inside one clip."""
    hh, hv, tf, epsg = read_clip(path)
    geom = gpd.GeoSeries([polygon], crs=4326).to_crs(epsg).iloc[0]
    if BUFFER_M:
        geom = geom.buffer(-BUFFER_M)
    # True where the pixel centre lies inside the polygon.
    inside = geometry_mask([geom], out_shape=hh.shape, transform=tf, invert=True)
    ok = inside & np.isfinite(hh) & np.isfinite(hv)
    rec = {"n_pixels": int(inside.sum()), "n_valid": int(ok.sum())}
    if ok.any():
        hh_lin, hv_lin = hh[ok].astype("f8").mean(), hv[ok].astype("f8").mean()
        rec.update(hh_lin=hh_lin, hv_lin=hv_lin, hh_db=to_db(hh_lin),
                   hv_db=to_db(hv_lin), ratio_db=to_db(hh_lin / hv_lin))   # ratio of MEANS
    return rec


def block_mean(a, k):
    """Average k x k blocks ignoring NaN; trims edges that don't fill a block."""
    if k <= 1:
        return a
    ny, nx = (a.shape[0] // k) * k, (a.shape[1] // k) * k
    return np.nanmean(a[:ny, :nx].reshape(ny // k, k, nx // k, k), axis=(1, 3))


def map_layers(path):
    """dB layers for mapping one clip, plus its extent in km and EPSG."""
    hh, hv, tf, epsg = read_clip(path)
    hh, hv = block_mean(hh, MAP_LOOKS), block_mean(hv, MAP_LOOKS)    # multilook in LINEAR
    left, top = tf.c, tf.f
    extent = np.array([left, left + hh.shape[1] * MAP_LOOKS * tf.a,
                       top + hh.shape[0] * MAP_LOOKS * tf.e, top]) / 1e3
    return {"HHHH": to_db(hh), "HVHV": to_db(hv), "ratio": to_db(hh / hv),
            "extent": extent, "epsg": epsg}

## 5. Spatial means for every pass × field

In [ ]:
rows = []
for _, ps in passes.iterrows():
    for _, fld in fields.iterrows():
        rec = {k: ps[k] for k in ("granule", "track", "frame", "direction", "acq_utc")}
        rec["field"] = fld.field
        path = clip_path(ps.granule, fld.field)
        if not fld.has_polygon:
            rec["note"] = "no polygon"
        elif path is None:
            rec["note"] = "no clip"
        else:
            rec.update(field_mean(path, fld.geometry))
        rows.append(rec)

means = pd.DataFrame(rows).sort_values(["field", "acq_utc"]).reset_index(drop=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
means.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV.name}: {len(means)} rows")
means.drop(columns=["granule"]).round(2)

## 6. Time series of the field means

One column per **track** — different viewing geometries, never joined.

In [ ]:
ts = means.dropna(subset=["hh_db"]) if "hh_db" in means else means.iloc[0:0]
QUANTITIES = [("hh_db", "HHHH [dB]"), ("hv_db", "HVHV [dB]"), ("ratio_db", "HH − HV [dB]")]
COLORS = dict(zip(FIELDS_2026, ["#1f77b4", "#17becf", "#ff7f0e", "#2ca02c", "#d62728"]))
tracks = sorted(ts.track.unique())

fig, axes = plt.subplots(len(QUANTITIES), max(len(tracks), 1),
                         figsize=(7 * max(len(tracks), 1), 12), sharex=True, squeeze=False)
for j, trk in enumerate(tracks):
    sub = ts[ts.track == trk]
    for i, (col, label) in enumerate(QUANTITIES):
        ax = axes[i][j]
        for fld, g in sub.groupby("field"):
            ax.plot(g.acq_utc, g[col], "o-", ms=6, lw=1.8, color=COLORS.get(fld), label=fld)
        ax.grid(alpha=0.3)
        if j == 0:
            ax.set_ylabel(label)
        if i == 0:
            ax.set_title(f"track {trk} ({sub.direction.iloc[0]}) — {sub.granule.nunique()} passes")

pad = pd.Timedelta(days=5)
for ax in axes.flat:
    ax.set_xlim(ts.acq_utc.min() - pad, ts.acq_utc.max() + pad)     # stop auto-padding
for ax in axes[-1]:
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.set_xlabel("acquisition date (UTC)")
h, l = axes[0][0].get_legend_handles_labels()
fig.legend(h, l, loc="center left", bbox_to_anchor=(1.0, 0.5), title="field")
fig.suptitle(f"Field-mean backscatter, {YEAR} (polygon buffer {BUFFER_M} m)", y=1.0)
fig.tight_layout()
fig.savefig(TS_PNG, dpi=120, bbox_inches="tight")
plt.show()

## 7. Maps — one figure per pass

Rows = fields (order of `FIELDS_2026`), columns = HHHH / HVHV / HH−HV.
Cyan = polygon, white triangles = gauges.

In [ ]:
PANELS = [("HHHH", "HHHH [dB]", "gray"), ("HVHV", "HVHV [dB]", "gray"),
          ("ratio", "HH − HV [dB]", "magma")]
OUTLINE = [pe.withStroke(linewidth=3, foreground="black")]        # readable labels


def plot_maps(ps):
    """One figure for one pass (a row of `passes`): a row per field, three columns."""
    fig, axes = plt.subplots(len(fields), 3, figsize=(17, 4.6 * len(fields)), squeeze=False)
    ims, epsg = [None] * 3, None
    for i, (_, fld) in enumerate(fields.iterrows()):
        path = clip_path(ps.granule, fld.field)
        row_label = fld.field + ("" if fld.has_polygon else "\n(no polygon)")
        axes[i][0].set_ylabel(f"{row_label}\nnorthing [km]", fontweight="bold")
        if path is None:                                 # field not covered by this pass
            for ax in axes[i]:
                ax.text(0.5, 0.5, "no data this pass", ha="center", transform=ax.transAxes)
                ax.set_xticks([]); ax.set_yticks([])
            continue
        w = map_layers(path); epsg = w["epsg"]
        to_km = lambda gs: gs.to_crs(epsg).scale(1e-3, 1e-3, origin=(0, 0))
        poly_km = to_km(gpd.GeoSeries([fld.geometry], crs=4326)) if fld.has_polygon else None
        gpts = gauges[gauges.field_name == fld.field]
        gpts_km = to_km(gpts.geometry)
        for j, (key, label, cmap) in enumerate(PANELS):
            ax = axes[i][j]
            ims[j] = ax.imshow(w[key], extent=w["extent"], cmap=cmap, interpolation="nearest",
                               vmin=VLIM[key][0], vmax=VLIM[key][1])
            if poly_km is not None:
                poly_km.boundary.plot(ax=ax, color="#00e5ff", lw=1.8)
            ax.plot(gpts_km.x, gpts_km.y, "^", ms=10, mfc="white", mec="black", mew=1.2)
            for gid, p in zip(gpts.gauge_id, gpts_km):
                ax.annotate(gid, (p.x, p.y), xytext=(6, 6), textcoords="offset points",
                            color="white", fontsize=11, fontweight="bold", path_effects=OUTLINE)
            ax.set_title(label if i == 0 else "")
            ax.tick_params(labelsize=11)
            ax.ticklabel_format(useOffset=False, style="plain")   # no "+3.817e3" offsets
            ax.xaxis.set_major_locator(plt.MaxNLocator(5))        # stop labels running together
    for j, (key, label, _) in enumerate(PANELS):         # one colourbar per column
        if ims[j] is not None:
            fig.colorbar(ims[j], ax=axes[:, j].tolist(), orientation="horizontal",
                         pad=0.03, fraction=0.025, aspect=40, label=label)
    fig.canvas.draw()                                    # title just above the first row
    top = max(ax.get_position().y1 for ax in axes[0])
    fig.suptitle(f"{ps.acq_utc:%Y-%m-%d %H:%M} UTC — track {ps.track} {ps.direction}"
                 f" — {MAP_LOOKS}x{MAP_LOOKS} looks — km, EPSG:{epsg}", y=top + 0.025)
    return fig


for k in MAP_DATES:                                      # show selected passes inline
    if k < len(passes):
        plot_maps(passes.iloc[k]); plt.show()

if SAVE_ALL_MAPS:                                        # one PNG per pass
    MAP_DIR.mkdir(parents=True, exist_ok=True)
    for _, ps in passes.iterrows():
        fig = plot_maps(ps)
        fig.savefig(MAP_DIR / f"{ps.acq_utc:%Y%m%d}_T{ps.track}_{ps.direction}.png",
                    dpi=110, bbox_inches="tight")
        plt.close(fig)
    print(f"saved {len(passes)} figures to {MAP_DIR}")